# Step 2: Filter notebook

In [1]:
import geopandas as gpd
import pandas as pd
from functools import reduce
import osmnx as ox
from shapely.geometry import Point
from matplotlib import pyplot as plt
import rasterio

import os

from method_a_buffer import extract_buffer_feature

import geopandas as gpd
import folium
from folium import Choropleth, CircleMarker, GeoJson
import osmnx as ox
import branca.colormap as cm

from shapely.geometry import Point, box, LineString

# Display the map in the notebook
from IPython.display import display
# Display all columns in a dataframe
pd.set_option('display.max_columns', None)

## Imports

#### Import des segments

In [2]:
operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

input_file_path = '../../Data/input/attributs'
output_step1_path='../../Data/output/step-1'
output_step2_path='../../Data/output/step-2'
output_step3_path='../../Data/output/step-3'

save_filtered_attributes = False

# Load segments GeoDataFrame (with 'segment_id')
print("Loading pedestrian segments...")
# reLoad pedestrian segments
segmented_net = gpd.read_parquet(os.path.join(output_step1_path, "step1_pedestrian_segments.parquet"))
segmented_net = segmented_net.to_crs(operation_crs)

# Define a function to save the filtered data
def save(save_filtered_attributes, row, gdf, attribute):
    if save_filtered_attributes:
        if row['save_format'] == 'parquet':
            if not os.path.exists(f'{output_step2_path}/gpkg_attributs'):
                os.makedirs(f'{output_step2_path}/gpkg_attributs')
            gdf.to_parquet(f"{output_step2_path}/parquet_attributs/{attribute}.parquet")
            print(f"Filtered data saved for attribute: {attribute} in format: {row['save_format']}")
        elif row['save_format'] == 'gpkg':
            if not os.path.exists(f'{output_step2_path}/parquet_attributs'):
                os.makedirs(f'{output_step2_path}/parquet_attributs')
            gdf.to_file(f"{output_step2_path}/gpkg_attributs/{attribute}.gpkg", driver="GPKG")
            print(f"Filtered data saved for attribute: {attribute} in format: {row['save_format']}")
        else:
            if not os.path.exists(f'{output_step2_path}/csv_attributs'):
                os.makedirs(f'{output_step2_path}/csv_attributs')
            gdf.to_csv(f"{output_step2_path}/csv_attributs/{attribute}.csv", index=False)
            print(f"Warning: Unknown save format {row['save_format']} for attribute {attribute}. Data saved as csv.")
    else:
        print("Note : Save option is disabled.")

Loading pedestrian segments...


**Connectivité du réseau**

In [3]:
import geopandas as gpd
import pandas as pd
import numpy as np
import networkx as nx
from shapely.geometry import Point

def compute_connectivity_metrics(
    segmented_net: gpd.GeoDataFrame,
    buffer_m: int = 50,
    compute_betweenness: bool = False,
    betweenness_k: int | None = None,   # échantillonnage pour accélérer (k=100 par ex.)
    crs_meter_epsg: int | None = None,  # si ton GDF est en degrés, projette d'abord (ex. 2056)
) -> gpd.GeoDataFrame:
    """
    Ajoute des métriques de connectivité par segment.

    Paramètres
    ----------
    segmented_net : GeoDataFrame avec colonnes u, v, key, segment_id, geometry, (facultatif: length_m)
    buffer_m : rayon pour les métriques locales
    compute_betweenness : calcule une betweenness approx. (moyenne des betweenness des deux nœuds)
    betweenness_k : échantillonnage de noeuds pour accélérer betweenness (NetworkX), None = exact
    crs_meter_epsg : si fourni et si le CRS n’est pas métrique, reprojette pour les buffers

    Retour
    ------
    GeoDataFrame enrichi avec colonnes:
      - conn_mean_degree, conn_deadend_flag, conn_intersection_flag
      - conn_nodes_in_buffer, conn_edges_in_buffer
      - conn_intersections_in_buffer, conn_deadends_in_buffer
      - conn_branching_in_buffer, conn_beta_local
      - conn_betweenness (si compute_betweenness=True)
    """

    gdf = segmented_net.copy()

    # --- Sécurité CRS métrique pour les buffers
    if crs_meter_epsg is not None and (gdf.crs is None or not gdf.crs.is_projected):
        gdf = gdf.to_crs(crs_meter_epsg)

    # Longueur en mètres si manquante (suppose CRS métrique)
    if "length_m" not in gdf.columns:
        gdf["length_m"] = gdf.geometry.length

    # --- Construire le graphe (MultiGraph pour respecter les multi-arêtes)
    G = nx.MultiGraph()
    # On ajoute les arêtes avec attributs utiles
    for r in gdf.itertuples(index=False):
        G.add_edge(getattr(r, "u"), getattr(r, "v"),
                   key=getattr(r, "key"),
                   segment_id=getattr(r, "segment_id"),
                   length=getattr(r, "length_m"))

    # --- Degrés des nœuds
    node_degree = dict(G.degree())
    nx.set_node_attributes(G, node_degree, "degree")

    # --- Extraire une table des nœuds (u/v) avec géométrie (depuis les extrémités des segments)
    # NB: si tes IDs u/v proviennent d’OSMnx, c’est cohérent ; sinon, on reconstruit via endpoints.
    node_rows = []
    for r in gdf.itertuples(index=False):
        geom = getattr(r, "geometry")
        # start / end
        x0, y0 = geom.coords[0]
        x1, y1 = geom.coords[-1]
        node_rows.append({"node": getattr(r, "u"), "geometry": Point(x0, y0)})
        node_rows.append({"node": getattr(r, "v"), "geometry": Point(x1, y1)})
    nodes_gdf = gpd.GeoDataFrame(node_rows, geometry="geometry", crs=gdf.crs)
    # Conserver une géométrie par node ID (si multiples, on prend la première)
    nodes_gdf = nodes_gdf.drop_duplicates(subset="node", keep="first").reset_index(drop=True)
    # Ajouter le degree
    nodes_gdf["degree"] = nodes_gdf["node"].map(node_degree).fillna(0).astype(int)


    n_sindex = nodes_gdf.sindex

    # Containers résultats par segment
    out = {
        "segment_id": [],
        "conn_mean_degree": [],
        "conn_deadend_flag": [],
        "conn_intersection_flag": [],
        "conn_nodes_in_buffer": [],
        "conn_edges_in_buffer": [],
        "conn_intersections_in_buffer": [],
        "conn_deadends_in_buffer": [],
        "conn_branching_in_buffer": [],
        "conn_beta_local": [],
    }

    # --- (Optionnel) Betweenness des NOEUDS (approx) pour reporter aux segments
    node_bet = None
    if compute_betweenness:
        # Graph simple pondéré par longueur (plus stable que MultiGraph pour centrality)
        H = nx.Graph()
        for u, v, data in G.edges(data=True):
            w = data.get("length", 1.0)
            if H.has_edge(u, v):
                if w < H[u][v]["weight"]:
                    H[u][v]["weight"] = w
            else:
                H.add_edge(u, v, weight=w)

    # Ici k doit être un int (nb de nœuds à échantillonner) ou None
    node_bet = nx.betweenness_centrality(
        H,
        k=betweenness_k,          # <-- ENTIER (ex. 200) ou None pour exact
        weight="weight",
        normalized=True,
        endpoints=False,
        seed=42                    # pour reproductibilité
    )

    # --- spatial index (on peut le garder)
    e_sindex = gdf.sindex
    n_sindex = nodes_gdf.sindex

    # --- Boucle segments (sans colonne _buffer)
    for r in gdf.itertuples(index=False):
        seg_id = getattr(r, "segment_id")
        u, v = getattr(r, "u"), getattr(r, "v")

        # buffer local (évite le problème d'attribut)
        buf = getattr(r, "geometry").buffer(buffer_m)

        minx, miny, maxx, maxy = buf.bounds
        query_geom = box(minx, miny, maxx, maxy)

        deg_u = node_degree.get(u, 0)
        deg_v = node_degree.get(v, 0)
        mean_deg = (deg_u + deg_v) / 2

        deadend_flag = (deg_u == 1) or (deg_v == 1)
        intersection_flag = (deg_u >= 3) or (deg_v >= 3)

        # --- Nœuds dans le buffer
        cand_nodes_idx = list(n_sindex.query(query_geom, predicate='intersects'))
        nodes_in_buf = nodes_gdf.iloc[cand_nodes_idx]
        nodes_in_buf = nodes_in_buf[nodes_in_buf.geometry.intersects(buf)]

        nb_nodes = len(nodes_in_buf)
        nb_intersections = int((nodes_in_buf["degree"] >= 3).sum())
        nb_deadends = int((nodes_in_buf["degree"] == 1).sum())
        branching = int(((nodes_in_buf["degree"] - 2).clip(lower=0)).sum())

        # --- Arêtes dans le buffer (excluant soi-même)
        cand_edges_idx = list(e_sindex.query(query_geom, predicate='intersects'))
        edges_in_buf = gdf.iloc[cand_edges_idx]
        edges_in_buf = edges_in_buf[edges_in_buf.geometry.intersects(buf)]
        nb_edges = int(len(edges_in_buf) - 1)  # retirer le segment courant

        beta_local = nb_edges / max(1, nb_nodes)

        out["segment_id"].append(seg_id)
        out["conn_mean_degree"].append(mean_deg)
        out["conn_deadend_flag"].append(bool(deadend_flag))
        out["conn_intersection_flag"].append(bool(intersection_flag))
        out["conn_nodes_in_buffer"].append(int(nb_nodes))
        out["conn_edges_in_buffer"].append(int(nb_edges))
        out["conn_intersections_in_buffer"].append(int(nb_intersections))
        out["conn_deadends_in_buffer"].append(int(nb_deadends))
        out["conn_branching_in_buffer"].append(int(branching))
        out["conn_beta_local"].append(float(beta_local))

    metrics = pd.DataFrame(out)

    # --- Betweenness reportée au segment (moyenne des deux nœuds) si demandé
    if compute_betweenness and node_bet is not None:
        bet_vals = []
        for r in gdf.itertuples(index=False):
            u, v = getattr(r, "u"), getattr(r, "v")
            b = 0.5 * (node_bet.get(u, 0.0) + node_bet.get(v, 0.0))
            bet_vals.append(b)
        gdf["conn_betweenness"] = bet_vals

    # Fusion finale
    gdf = gdf.merge(metrics, on="segment_id", how="left")

    return gdf


In [4]:
# Ensure geometry column is set to avoid spatial index errors
segmented_net = segmented_net.set_geometry("geometry")
print(segmented_net.columns)

Index(['Largeur', 'Partage_us', 'Objet', 'Revetement', 'Vitesse', 'Zone_mod',
       'Commune', 'Type', 'Pente', 'Classe', 'Nom_voie', 'Franchisse',
       'PP_Feux', 'SHAPE_Leng', 'geometry', 'length', 'segment_id'],
      dtype='object')


In [5]:
def add_uv_columns(gdf):
    gdf = gdf.copy()
    gdf["u"] = gdf.geometry.apply(lambda g: hash(g.coords[0]))   # noeud départ
    gdf["v"] = gdf.geometry.apply(lambda g: hash(g.coords[-1]))  # noeud arrivée
    gdf["key"] = 0  # pas de multi-arêtes, donc clé unique
    return gdf

segmented_net = add_uv_columns(segmented_net)

In [6]:

segmented_net_metrics = compute_connectivity_metrics(
    segmented_net,
    buffer_m=100,
    compute_betweenness=True,
    betweenness_k=200
)

segmented_net_metrics['filtered'] = 1

In [7]:
segmented_net_metrics

,Largeur,Partage_us,Objet,Revetement,Vitesse,Zone_mod,Commune,Type,Pente,Classe,Nom_voie,Franchisse,PP_Feux,SHAPE_Leng,geometry,length,segment_id,u,v,key,length_m,conn_betweenness,conn_mean_degree,conn_deadend_flag,conn_intersection_flag,conn_nodes_in_buffer,conn_edges_in_buffer,conn_intersections_in_buffer,conn_deadends_in_buffer,conn_branching_in_buffer,conn_beta_local,filtered
0,Très étroit,Aucun,Trottoir,Béton bitumineux,50,None,Thônex,Trottoir,2.6,tertiair,Chemin des Tourterelles,Sans,None,18.611510,"LINESTRING (2505952.43 1117556.983, 2505957.52...",18.611510,000000,-6930693752093156798,-5320831420533533266,0,18.611510,0.000000,1.0,True,False,36,34,4,16,4,0.944444,1
1,Pas de trottoir,Aucun,Trottoir,Béton bitumineux,50,None,Thônex,Passage piéton,0.3,tertiair,Voie Marguerite-MARMOUD,Sans,Oui,17.373877,"LINESTRING (2504875.759 1116854.188, 2504891.4...",17.373877,000001,-5650924595398224121,5479560796140918503,0,17.373877,0.000000,2.0,False,False,73,81,19,20,21,1.109589,1
2,Large,Mixité vélos,Trottoir,Pavés,50,None,Genève-Cité,Trottoir,0.3,tertiair,Rue de la Monnaie,Sans,None,57.198671,"LINESTRING (2500026.558 1117819.302, 2500041.9...",50.000000,000002,-4271552768887947945,-8292225703009643650,0,50.000000,0.000000,2.0,False,False,118,147,51,31,57,1.245763,1
3,Large,Mixité vélos,Trottoir,Pavés,50,None,Genève-Cité,Trottoir,0.3,tertiair,Rue de la Monnaie,Sans,None,57.198671,"LINESTRING (2500045.896 1117773.338, 2500047.9...",7.198671,000003,-8292225703009643650,8047137376149879673,0,7.198671,0.000000,1.5,True,False,97,125,47,23,52,1.288660,1
4,Moyen,Aucun,Trottoir,Pavés,0,Zone piétonne,Lancy,Trottoir,1.5,tertiair,Esplanade de Pont-Rouge,Sans,None,45.125485,"LINESTRING (2498574.627 1115881.289, 2498530.0...",45.125485,000004,-6541324296344382923,-5364403646111035711,0,45.125485,0.000000,1.0,True,False,83,102,33,20,42,1.228916,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134065,Pas de trottoir,None,None,None,0,None,Plan-les-Ouates,Voie de service,4.0,secondai,None,None,None,6.136427,"LINESTRING (2498917.956 1113532.063, 2498922.7...",6.136427,134065,3375169246876258826,8769006445882061809,0,6.136427,0.000000,3.0,False,True,38,43,9,9,9,1.131579,1
134066,Pas de trottoir,None,None,None,0,None,Lancy,Voie de service,7.8,secondai,None,None,None,9.836936,"LINESTRING (2497663.661 1116834.636, 2497661.4...",9.836936,134066,-8130421142391585299,-1351949987365497831,0,9.836936,0.000000,3.0,False,True,81,83,19,31,20,1.024691,1
134067,Pas de trottoir,None,Parking,Béton bitumineux,50,None,Carouge,Voie de service,1.1,secondai,None,None,None,9.957378,"LINESTRING (2499292.972 1115548.703, 2499294.5...",9.957378,134067,-7327938675199793618,-401580201136213416,0,9.957378,0.000015,3.0,False,True,96,121,37,23,45,1.260417,1
134068,Pas de trottoir,None,None,None,0,None,Vernier,Voie de service,0.3,secondai,None,None,None,8.728223,"LINESTRING (2496898.393 1119087.889, 2496904.1...",8.728223,134068,-3881790742168696375,2435142573084777007,0,8.728223,0.000000,3.0,False,True,26,33,12,3,13,1.269231,1


In [8]:
#save segmented net in parquet
segmented_net_metrics.to_parquet(os.path.join(output_step2_path, "parquet_attributs/connectivite.parquet"))

#save segmented net with metrics in geopackage
segmented_net_metrics.to_file(os.path.join(output_step2_path, "gpkg_attributs/connectivite.gpkg"), driver="GPKG")


**Ratio des surfaces piétones**

In [28]:
# Analyse de la part d’espace par usage (trottoirs, pistes cyclables, bus, chaussée)
# à partir du réseau routier linéaire et des objets surfaciques du cadastre routier (SITG)

import geopandas as gpd
import pandas as pd
from shapely.geometry import LineString, box
import matplotlib.pyplot as plt
import folium
import branca.colormap as cm
import warnings

# Filtrer certains warnings gênants
warnings.filterwarnings("ignore", category=UserWarning, module="pyproj")
warnings.filterwarnings("ignore", category=RuntimeWarning, module="pyogrio")
warnings.filterwarnings("ignore", category=DeprecationWarning, message=".*unary_union.*")

# -------------------------------------------------
# 1. Chargement et préparation des données
# -------------------------------------------------
print("Chargement des shapefiles…")
lines = segmented_net.to_crs("EPSG:2056")
surfaces = gpd.read_file(f"{input_file_path}/domaine_routier/CAD_DOMAINE_ROUTIER-SHP/CAD_DOMAINE_ROUTIER.shp").to_crs("EPSG:2056")

# Nettoyage des géométries surfaciques
surfaces = surfaces[surfaces.is_valid & surfaces.geometry.notnull()].copy()
surfaces = surfaces[surfaces.geometry.area > 1].copy()



Chargement des shapefiles…


In [29]:
surfaces.OBJET.value_counts()

OBJET
Surface latérale                   13544
Ilot latéral                        9221
Trottoir                            8704
Chaussée                            8132
Ilot circulation                    5350
Chemin                              3787
Espace de stationnement             1914
Piste cyclable                       682
Parking                              511
Site propre transport en commun      283
Name: count, dtype: int64

In [30]:

# -------------------------------------------------
# 1. Chargement et préparation des données
# -------------------------------------------------
print("Chargement des shapefiles…")
lines = segmented_net.to_crs("EPSG:2056")
surfaces = gpd.read_file(f"{input_file_path}/domaine_routier/CAD_DOMAINE_ROUTIER-SHP/CAD_DOMAINE_ROUTIER.shp").to_crs("EPSG:2056")

# Nettoyage des géométries surfaciques
surfaces = surfaces[surfaces.is_valid & surfaces.geometry.notnull()].copy()
surfaces = surfaces[surfaces.geometry.area > 1].copy()

# -------------------------------------------------
# 2. Buffer de 100 m autour des lignes
# -------------------------------------------------
print("Création des zones tampons autour des lignes…")
lines = lines.reset_index().rename(columns={"index": "line_index"})
lines["buffer"] = lines.geometry.buffer(75, cap_style="round")
lines_buffered = lines.set_geometry("buffer")

# -------------------------------------------------
# 3. Intersections surfaciques / linéaires
# -------------------------------------------------
print("Calcul de l’intersection entre surfaces et buffers de lignes…")
intersected = gpd.overlay(surfaces, lines_buffered, how="intersection")
intersected["area"] = intersected.geometry.area

# -------------------------------------------------
# 4. Agrégation des surfaces par ligne et usage
# -------------------------------------------------
print("Agrégation des surfaces par tronçon…")
intersected = intersected[intersected["area"] > 0].copy()
grouped = intersected.groupby(["line_index", "OBJET"])["area"].sum().unstack(fill_value=0).reset_index()

# Totaux + ratios
grouped["total"] = grouped.drop(columns=["line_index"]).sum(axis=1)
if "t" in grouped.columns:
    grouped["ratio_trottoir"] = grouped["Trottoir"] / grouped["total"]
if "pc" in grouped.columns:
    grouped["ratio_cyclable"] = grouped["Piste cyclable"] / grouped["total"]
if "tc" in grouped.columns:
    grouped["ratio_tc"] = grouped["Site propre transport en commun"] / grouped["total"]
if "c" in grouped.columns:
    grouped["ratio_chaussée"] = grouped["Chausée"] / grouped["total"]

# -------------------------------------------------
# 5. Fusion avec les lignes originales
# -------------------------------------------------
print("Fusion avec les données linéaires originales…")
lines = lines.merge(grouped, on="line_index", how="left")
lines = lines.set_geometry("geometry").drop(columns="buffer")






Chargement des shapefiles…
Création des zones tampons autour des lignes…
Calcul de l’intersection entre surfaces et buffers de lignes…
Agrégation des surfaces par tronçon…
Fusion avec les données linéaires originales…


In [31]:
surfaces

,OBJECTID,COMMUNE,OBJET,REVETEMENT,NIVEAU,SHAPE_AREA,SHAPE_LEN,geometry
0,41,Pregny-Chambésy,Surface latérale,Béton,-1,14.075099,113.610981,"POLYGON ((2500691.463 1122036.697, 2500687.392..."
1,42,Pregny-Chambésy,Chaussée,Béton bitumineux,-1,195.315776,117.346954,"POLYGON ((2500689.977 1122032.02, 2500685.904 ..."
2,43,Pregny-Chambésy,Trottoir,Béton bitumineux,-1,91.127450,93.386230,"POLYGON ((2500727.863 1122042.732, 2500726.002..."
3,44,Grand-Saconnex,Trottoir,Béton bitumineux,0,119.505563,218.750903,"POLYGON ((2498021.786 1121226.771, 2498023.388..."
4,45,Grand-Saconnex,Chaussée,Béton bitumineux,0,267.332292,223.444158,"POLYGON ((2498097.807 1121164.718, 2498098.446..."
...,...,...,...,...,...,...,...,...
52592,52212,Pregny-Chambésy,Chaussée,Béton bitumineux,0,1134.658253,218.576665,"POLYGON ((2500772.129 1122231.261, 2500772.119..."
52593,52213,Bellevue,Surface latérale,Prairie,0,283.839029,92.829144,"POLYGON ((2500704.583 1122495.566, 2500704.526..."
52594,52214,Bellevue,Surface latérale,Arbustes,0,404.073432,92.530211,"POLYGON ((2500696.886 1122496.532, 2500677.64 ..."
52595,52215,Bellevue,Ilot latéral,Arbustes,0,3211.401017,709.691922,"POLYGON ((2500683.84 1122492.633, 2500693.614 ..."


In [32]:
grouped

OBJET,line_index,Chaussée,Chemin,Espace de stationnement,Ilot circulation,Ilot latéral,Parking,Piste cyclable,Site propre transport en commun,Surface latérale,Trottoir,total
0,0,2831.518163,97.649888,0.000000,0.000000,0.000000,0.000000,4.521063,0.000000,636.688797,1130.017083,4700.394995
1,1,3137.044133,503.632711,0.000000,52.429632,138.886264,0.000000,551.030866,0.000000,168.379907,968.068209,5519.471723
2,2,5042.326093,1061.332316,0.000000,158.215473,1291.915956,0.000000,0.000000,503.375740,0.000000,5234.886961,13292.052538
3,3,4431.585390,1081.402470,0.000000,145.656420,498.726133,0.000000,0.000000,211.583622,0.000000,3943.231337,10312.185371
4,4,621.772182,1381.830215,0.000000,0.000000,461.141765,0.000000,0.000000,83.656729,170.637230,2550.067315,5269.105436
...,...,...,...,...,...,...,...,...,...,...,...,...
127909,134063,376.094227,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,54.451154,430.545381
127910,134064,648.609896,116.020978,0.000000,0.000000,0.000000,0.000000,93.135748,0.000000,0.000000,153.843603,1011.610226
127911,134065,914.647744,70.853700,0.000000,0.000000,0.000000,0.000000,49.405258,0.000000,0.000000,141.256348,1176.163051
127912,134066,4684.650344,0.000000,285.749302,240.339399,102.418697,0.000000,251.878234,0.000000,207.505811,1575.388228,7347.930016


In [33]:
lines

,line_index,Largeur,Partage_us,Objet,Revetement,Vitesse,Zone_mod,Commune,Type,Pente,Classe,Nom_voie,Franchisse,PP_Feux,SHAPE_Leng,geometry,length,segment_id,u,v,key,Chaussée,Chemin,Espace de stationnement,Ilot circulation,Ilot latéral,Parking,Piste cyclable,Site propre transport en commun,Surface latérale,Trottoir,total
0,0,Très étroit,Aucun,Trottoir,Béton bitumineux,50,None,Thônex,Trottoir,2.6,tertiair,Chemin des Tourterelles,Sans,None,18.611510,"LINESTRING (2505952.43 1117556.983, 2505957.52...",18.611510,000000,-6930693752093156798,-5320831420533533266,0,2831.518163,97.649888,0.000000,0.000000,0.000000,0.000000,4.521063,0.000000,636.688797,1130.017083,4700.394995
1,1,Pas de trottoir,Aucun,Trottoir,Béton bitumineux,50,None,Thônex,Passage piéton,0.3,tertiair,Voie Marguerite-MARMOUD,Sans,Oui,17.373877,"LINESTRING (2504875.759 1116854.188, 2504891.4...",17.373877,000001,-5650924595398224121,5479560796140918503,0,3137.044133,503.632711,0.000000,52.429632,138.886264,0.000000,551.030866,0.000000,168.379907,968.068209,5519.471723
2,2,Large,Mixité vélos,Trottoir,Pavés,50,None,Genève-Cité,Trottoir,0.3,tertiair,Rue de la Monnaie,Sans,None,57.198671,"LINESTRING (2500026.558 1117819.302, 2500041.9...",50.000000,000002,-4271552768887947945,-8292225703009643650,0,5042.326093,1061.332316,0.000000,158.215473,1291.915956,0.000000,0.000000,503.375740,0.000000,5234.886961,13292.052538
3,3,Large,Mixité vélos,Trottoir,Pavés,50,None,Genève-Cité,Trottoir,0.3,tertiair,Rue de la Monnaie,Sans,None,57.198671,"LINESTRING (2500045.896 1117773.338, 2500047.9...",7.198671,000003,-8292225703009643650,8047137376149879673,0,4431.585390,1081.402470,0.000000,145.656420,498.726133,0.000000,0.000000,211.583622,0.000000,3943.231337,10312.185371
4,4,Moyen,Aucun,Trottoir,Pavés,0,Zone piétonne,Lancy,Trottoir,1.5,tertiair,Esplanade de Pont-Rouge,Sans,None,45.125485,"LINESTRING (2498574.627 1115881.289, 2498530.0...",45.125485,000004,-6541324296344382923,-5364403646111035711,0,621.772182,1381.830215,0.000000,0.000000,461.141765,0.000000,0.000000,83.656729,170.637230,2550.067315,5269.105436
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134065,134065,Pas de trottoir,None,None,None,0,None,Plan-les-Ouates,Voie de service,4.0,secondai,None,None,None,6.136427,"LINESTRING (2498917.956 1113532.063, 2498922.7...",6.136427,134065,3375169246876258826,8769006445882061809,0,914.647744,70.853700,0.000000,0.000000,0.000000,0.000000,49.405258,0.000000,0.000000,141.256348,1176.163051
134066,134066,Pas de trottoir,None,None,None,0,None,Lancy,Voie de service,7.8,secondai,None,None,None,9.836936,"LINESTRING (2497663.661 1116834.636, 2497661.4...",9.836936,134066,-8130421142391585299,-1351949987365497831,0,4684.650344,0.000000,285.749302,240.339399,102.418697,0.000000,251.878234,0.000000,207.505811,1575.388228,7347.930016
134067,134067,Pas de trottoir,None,Parking,Béton bitumineux,50,None,Carouge,Voie de service,1.1,secondai,None,None,None,9.957378,"LINESTRING (2499292.972 1115548.703, 2499294.5...",9.957378,134067,-7327938675199793618,-401580201136213416,0,1034.938751,324.104122,0.000000,110.208610,146.010143,7890.011615,0.000000,0.000000,516.728437,265.021519,10287.023198
134068,134068,Pas de trottoir,None,None,None,0,None,Vernier,Voie de service,0.3,secondai,None,None,None,8.728223,"LINESTRING (2496898.393 1119087.889, 2496904.1...",8.728223,134068,-3881790742168696375,2435142573084777007,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Import des attributs -> à supprimer: redondance avec 2_1

In [14]:
attributs_info = pd.read_excel(f"{input_file_path}/attributs_info.xlsx", sheet_name="attributs_info")

In [15]:
attributs_info

,Class,meta_attribute,attribute,include_in_index,attribute_source_path,initial_weight,impact_attribut,proportional_weight_col_dtype,file_name,geometry_type,method,method_desc,how,value_column,buffer_size,filter_column,filter_values,crs,save_format,Unnamed: 19,check
0,Agrément,vegetation,arbre_isole,True,arbre_isole,0.5,favorable,float,SIPV_ICA_ARBRE_ISOLE-SHP/SIPV_ICA_ARBRE_ISOLE.shp,point,A,Buffer du segment,sum,D_COURONNE,10,filtered,1,2056,parquet,NaN,NaN
1,Agrément,vegetation,espace_vert,True,domaine_routier,0.5,favorable,NaN,CAD_DOMAINE_ROUTIER-SHP/CAD_DOMAINE_ROUTIER.shp,polygon,A,Buffer du segment,area_ratio,NaN,10,filtered,1,2056,parquet,NaN,NaN
2,Sécurité,accident,accident,True,accident,0.3,defavorable,NaN,OTC_ACCIDENTS-SHP/OTC_ACCIDENTS.shp,point,A,Buffer du segment,count,NaN,10,filtered,1,2056,parquet,NaN,NaN
3,NaN,toilette,toilette,False,NaN,0.5,favorable,NaN,VDG_WC_PUBLIC-SHP/VDG_WC_PUBLIC.shp,point,A,Buffer du segment,count,NaN,10,filtered,1,2056,parquet,NaN,NaN
4,Sécurité,traffic,zone_apaisee,True,vitesse,0.5,favorable,NaN,OTC_LIMITATIONS_VITESSE-SHP/OTC_LIMITATIONS_VI...,polygon,A,Buffer du segment,area_ratio,NaN,10,filtered,1,2056,parquet,NaN,NaN
5,Sécurité,traffic,zone_pietonne,True,vitesse,1.0,favorable,NaN,OTC_ZONE_MODERATION_TRAFIC-SHP/OTC_ZONE_MODERA...,polygon,A,Buffer du segment,area_ratio,NaN,10,filtered,1,2056,parquet,NaN,NaN
6,Sécurité,traffic,vitesse,True,vitesse,0.6,defavorable,NaN,OTC_LIMITATIONS_VITESSE-SHP/OTC_LIMITATIONS_VI...,polygon,A,Buffer du segment,area_ratio,NaN,10,filtered,1,2056,parquet,NaN,NaN
7,Sécurité,largeur trottoir,ratio_trottoir,True,domaine_routier,0.5,favorable,float,CAD_DOMAINE_ROUTIER-SHP/CAD_DOMAINE_ROUTIER.shp,polygon,A,Buffer du segment,area_ratio,NaN,10,filtered,1,2056,parquet,NaN,NaN
8,Attractivité,eau,eau,True,eau,0.5,favorable,NaN,LCE_GRAPHE_EAU-SHP/LCE_GRAPHE_EAU.shp,line,A,Buffer du segment,count,NaN,10,filtered,1,2056,parquet,NaN,ok
9,Attractivité,proximite,rez_actif,True,rez_actif,0.5,favorable,NaN,REG_ENTREPRISE_ETABLISSEMENT-SHP/REG_ENTREPRIS...,point,A,Buffer du segment,count,NaN,10,filtered,1,2056,parquet,NaN,NaN


**Attribut Accidents**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'accident'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.ANNEE > 2020, 'filtered'] = 1
gdf.loc[gdf.CONSEQ.isin(['Avec blessés graves', 'Avec tués']), 'filtered'] = 1
gdf.loc[gdf.VELOS == 1, 'filtered'] = 1
gdf.loc[gdf.VAE_25 == 1, 'filtered'] = 1
gdf.loc[gdf.VAE_45 == 1, 'filtered'] = 1
gdf.loc[gdf.PIETONS == 1, 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: accident
Filters applied
Proportion of features accident kept after filtering:
filtered
0    0.582074
1    0.417926
Name: proportion, dtype: float64
Note : Save option is disabled.


**Attribut Arbres isolés** (groupe Végétation)

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'arbre_isole'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)


Data initialized
Processing attribute: arbre_isole
Filters applied
Proportion of features arbre_isole kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Note : Save option is disabled.


**Attribut Espaces verts** (groupe Végétation)

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'espace_vert'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.REVETEMENT.isin(['Arbustes', 'Terre', 'Gazon','Grille gazon']), 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: espace_vert
Filters applied
Proportion of features espace_vert kept after filtering:
filtered
0    0.77455
1    0.22545
Name: proportion, dtype: float64
Note : Save option is disabled.


**Attribut Vitesse** (groupe Traffic)

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'vitesse'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.VITESSE > 30, 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: vitesse
Filters applied
Proportion of features vitesse kept after filtering:
filtered
1    0.671465
0    0.328535
Name: proportion, dtype: float64
Note : Save option is disabled.


**Attribut Zone pietonne** (groupe Traffic)

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'zone_pietonne'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.TYPE_ZONE.isin(['Zone piétonne', 'Zone de rencontre (20)']), 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Filters applied
Proportion of features zone_pietonne kept after filtering:
filtered
0    0.536313
1    0.463687
Name: proportion, dtype: float64
Note : Save option is disabled.


/Users/Helo/anaconda3/envs/actionsituee/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: organizePolygons() received an unexpected geometry.  Either a polygon with interior rings, or a polygon with less than 4 points, or a non-Polygon geometry.  Return arguments as a collection.
  return ogr_read(


**Attribut Zone apaisée** (groupe Traffic)

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'zone_apaisee'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.VITESSE <= 30, 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: zone_apaisee
Filters applied
Proportion of features zone_apaisee kept after filtering:
filtered
0    0.671465
1    0.328535
Name: proportion, dtype: float64
Note : Save option is disabled.


**Attribut Largeur trottoirs**

**Attribut Bruit**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'bruit'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.CAT_J > 2, 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: bruit
Filters applied
Proportion of features bruit kept after filtering:
filtered
0    0.887676
1    0.112324
Name: proportion, dtype: float64
Note : Save option is disabled.


**Attribut Proximité TP**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'tp'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: tp
Filters applied
Proportion of features tp kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Note : Save option is disabled.


**Attribut Rez Actifs**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'rez_actif'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
#keep if the column "BRANCHE" contains one of the following keywords (case insensitive)
keywords = ['commerce de détail', 'droguerie', 'enseignement', 'Hôpitaux', 'Hôtels', "Maisons", 'Paroisses et associations religieuses', 'Petits commerces', 'Petits supermarchés', 'Protection civile', 'Restaurants', 'Salons', 'Écoles'  ]
gdf.loc[gdf.BRANCHE.dropna().str.lower().apply(lambda val: any(kw.lower() in val for kw in keywords)), 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: rez_actif
Filters applied
Proportion of features rez_actif kept after filtering:
filtered
0    0.86085
1    0.13915
Name: proportion, dtype: float64
Note : Save option is disabled.


**Attribut Stationnement Genants**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'stationnement_genant'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: stationnement_genant
Filters applied
Proportion of features stationnement_genant kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Note : Save option is disabled.


**Attribut Proximité Aménités**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'amenite'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")


# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
#keep if the column "BRANCHE" contains one of the following keywords (case insensitive)
keywords = ['commerce de détail', 'droguerie', 'enseignement', 'Hôpitaux', 'Hôtels', "Maisons", 'Paroisses et associations religieuses', 'Petits commerces', 'Petits supermarchés', 'Protection civile', 'Restaurants', 'Salons', 'Écoles'  ]
gdf.loc[gdf.BRANCHE.dropna().str.lower().apply(lambda val: any(kw.lower() in val for kw in keywords)), 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: amenite
Filters applied
Proportion of features amenite kept after filtering:
filtered
0    0.86085
1    0.13915
Name: proportion, dtype: float64
Note : Save option is disabled.


**Attribut espaces ouverts**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'espaces_ouverts'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
gdf = gdf.to_crs(target_crs)
print(f"Processing attribute: {attribute}")


# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.TYPE == 'ESPACE PUBLIC', 'filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: espaces_ouverts
Filters applied
Proportion of features espaces_ouverts kept after filtering:
filtered
0    0.589171
1    0.410829
Name: proportion, dtype: float64
Note : Save option is disabled.


**Attribut Confort thermique**

In [13]:
import rasterio
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'temperature'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
raster_path = (f"{input_file_path}/{row['attribute_source_path']}/{row['file_name']}")
# Read raster with rasterio
with rasterio.open(raster_path) as src:
    # Read the raster data
    raster_data = src.read(1)  # Read first band
    raster_transform = src.transform
    raster_crs = src.crs
    
    # Filter out NoData values (-9999)
    valid_data = raster_data[raster_data != -9999]
    
    # Create a GeoDataFrame from the raster metadata
    gdf = gpd.GeoDataFrame({
        'temperature': raster_data.flatten(),
        'filtered': 0, # Initialize filtered column
        'geometry': None  # We'll add geometries if needed
    })
    gdf = gdf.set_crs(raster_crs)
print(f"Processing attribute: {attribute}")


# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

print("\nValue counts (first 10 most common values):")
print(gdf['temperature'].value_counts().head(10))
    
print("\nDescriptive statistics:")
print(gdf['temperature'].describe())

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: temperature
Filters applied
Proportion of features temperature kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64

Value counts (first 10 most common values):
temperature
 23.399401      319601
-9999.000000    311219
 23.399500       75534
 23.403999        4804
 23.403700        4749
 23.404100        4612
 23.403900        4472
 23.403799        4329
 23.403601        4053
 23.401600        3998
Name: count, dtype: int64

Descriptive statistics:
count    8.786745e+06
mean    -3.271767e+02
std      1.853357e+03
min     -9.999000e+03
25%      2.378310e+01
50%      2.855780e+01
75%      3.158280e+01
max      3.459120e+01
Name: temperature, dtype: float64
Note : Save option is disabled.


In [ ]:
# Boucle générique de chargement et prétraitement des couches attributs (sans doublons)
for _, row in attributs_info.iterrows():
    gdf = gpd.read_file(f"{input_file_path}/{row['attribute']}/{row['file_name']}")
    crs = row.get('crs', 2056)
    gdf = gdf.to_crs(crs)
    # Appliquer un filtre si besoin et si la colonne existe
    if row['filter_column'] and row['filter_values'] is not None and row['filter_column'] in gdf.columns:
        if isinstance(row['filter_values'], list):
            if row['attribute'] == 'rez_actif':
                gdf = gdf[gdf[row['filter_column']].dropna().str.lower().apply(lambda val: any(kw in val for kw in row['filter_values']))]
            else:
                gdf = gdf[gdf[row['filter_column']].isin(row['filter_values'])]
        else:
            gdf = gdf[gdf[row['filter_column']] == row['filter_values']]
    
    save_format = row.get('save_format', 'parquet')
    save_dir = f"{output_step2_path}/{'parquet_attributs' if save_format=='parquet' else 'gpkg_attributs'}"
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    out_file = f"{save_dir}/{row['attribute']}.{save_format if save_format=='parquet' else 'gpkg'}"
    if save_format == 'parquet':
        gdf.to_parquet(out_file)
    else:
        gdf.to_file(out_file, driver='GPKG')
    
    print(f"{row['attribute']} sauvegardé")

DataSourceError: ../../Data/input/arbre_isole/SIPV_ICA_ARBRE_ISOLE-SHP/SIPV_ICA_ARBRE_ISOLE.shp: No such file or directory